# Cross-Baseline Comparison v2 — V4 cohort
This aggregator concatenates the 6 baseline result CSVs (each saved by notebooks02–07 to `/lakehouse/default/Files/result/results_*.csv`) and builds the comparison table for the thesis. **Inputs are now from the V4 balanced cohort** — class collapse fixed, weekly features used directly. So per-class F1 (especially F1_D and F1_E) becomes meaningful, and W1/W2/W4 actually differ.

| Notebook | CSV |
|---|---|
| 02 — Fei & Yeung 2015 | `results_fei_yeung_2015.csv` |
| 03 — Alamri et al. 2019 | `results_alamri_2019.csv` |
| 04 — Dass et al. 2021 | `results_dass_2021.csv` |
| 05 — Xing & Du 2019 | `results_xing_du_2019.csv` |
| 06 — Basnet et al. 2022 | `results_basnet_2022.csv` |
| 07 — Aouarib et al. 2025 | `results_aouarib_2025.csv` |

In [3]:
import os, glob
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 55, Finished, Available, Finished, False)

In [4]:
# ============================================================
# 1) LOAD ALL BASELINE RESULTS
# ============================================================
PAPER_FILES = {
    "Fei&Yeung-2015":  "/lakehouse/default/Files/result/results_fei_yeung_2015.csv",
    "Alamri-2019":     "/lakehouse/default/Files/result/results_alamri_2019.csv",
    "Dass-2021":       "/lakehouse/default/Files/result/results_dass_2021.csv",
    "Xing&Du-2019":    "/lakehouse/default/Files/result/results_xing_du_2019.csv",
    "Basnet-2022":     "/lakehouse/default/Files/result/results_basnet_2022.csv",
    "Aouarib-2025":    "/lakehouse/default/Files/result/results_aouarib_2025.csv",
}

frames = []
for paper, path in PAPER_FILES.items():
    if not os.path.exists(path):
        print(f"  [skip] {paper}: {path} missing — run that notebook first")
        continue
    df = pd.read_csv(path)
    df.insert(0, "paper", paper)
    frames.append(df)
    print(f"  [ok]   {paper}: {len(df)} rows")

if not frames:
    raise RuntimeError("No baseline result CSVs found. Run notebooks 02-07 first.")

all_df = pd.concat(frames, ignore_index=True)
print(f"\nCombined shape: {all_df.shape}")
all_df.head()


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 56, Finished, Available, Finished, False)

  [ok]   Fei&Yeung-2015: 12 rows
  [ok]   Alamri-2019: 24 rows
  [ok]   Dass-2021: 6 rows
  [ok]   Xing&Du-2019: 6 rows
  [ok]   Basnet-2022: 48 rows
  [ok]   Aouarib-2025: 12 rows

Combined shape: (108, 19)


,paper,name,label,window,runtime_sec,accuracy,macro_f1,weighted_f1,f1_0,f1_1,auc,f1_A,f1_B,f1_C,f1_D,f1_E,auc_ovr,family,n_selected_feats
0,Fei&Yeung-2015,RNN,y_binary,1,28.70,0.855791,0.825333,0.850024,0.898272,0.752394,0.921555,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Fei&Yeung-2015,LSTM,y_binary,1,10.97,0.856986,0.826676,0.851211,0.899157,0.754194,0.924334,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Fei&Yeung-2015,RNN,y_binary,2,8.40,0.874315,0.850395,0.870645,0.910216,0.790574,0.929903,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Fei&Yeung-2015,LSTM,y_binary,2,9.10,0.875710,0.852583,0.872349,0.910972,0.794195,0.934128,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Fei&Yeung-2015,RNN,y_binary,4,10.49,0.880888,0.859328,0.877970,0.914400,0.804255,0.938217,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# ============================================================
# 2) UNIFY COLUMNS
# ============================================================
keep = ["paper", "name", "label", "window", "accuracy", "macro_f1", "weighted_f1",
        "auc", "auc_ovr", "f1_0", "f1_1", "f1_A", "f1_B", "f1_C", "f1_D", "f1_E",
        "runtime_sec", "family", "n_selected_feats"]
keep = [c for c in keep if c in all_df.columns]
unified = all_df[keep].copy()
unified = unified.replace({np.nan: None})
print(unified.head(15).to_string(index=False))


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 57, Finished, Available, Finished, False)

         paper             name    label  window  accuracy  macro_f1 weighted_f1       auc   auc_ovr      f1_0      f1_1      f1_A      f1_B      f1_C      f1_D      f1_E runtime_sec family n_selected_feats
Fei&Yeung-2015              RNN y_binary       1  0.855791  0.825333    0.850024  0.921555      None  0.898272  0.752394      None      None      None      None      None        28.7   None             None
Fei&Yeung-2015             LSTM y_binary       1  0.856986  0.826676    0.851211  0.924334      None  0.899157  0.754194      None      None      None      None      None       10.97   None             None
Fei&Yeung-2015              RNN y_binary       2  0.874315  0.850395    0.870645  0.929903      None  0.910216  0.790574      None      None      None      None      None         8.4   None             None
Fei&Yeung-2015             LSTM y_binary       2   0.87571  0.852583    0.872349  0.934128      None  0.910972  0.794195      None      None      None      None      None  

In [6]:
# ============================================================
# 3) SANITY CHECK — did the V4 cohort actually fix class collapse?
# ============================================================
# A degenerate model on V3 had macro_f1 = 0.1993 (predict-all-E) for 5-class.
# On V4, no row should be that low if the relabel + cohort filter worked.
deg = unified[(unified["label"] == "y_5class") & (unified["macro_f1"] < 0.25)]
if len(deg):
    print("WARNING — these 5-class rows look degenerate (Macro-F1 < 0.25):")
    print(deg.round(4).to_string(index=False))
else:
    print("OK — no degenerate 5-class rows. Class collapse is fixed by V4.")

# Also: F1_D and F1_E should both be > 0 in healthy runs
five_rows = unified[unified["label"] == "y_5class"]
if "f1_D" in five_rows.columns and "f1_E" in five_rows.columns:
    null_D = five_rows[five_rows["f1_D"].isna() | (five_rows["f1_D"] == 0)]
    null_E = five_rows[five_rows["f1_E"].isna() | (five_rows["f1_E"] == 0)]
    print(f"\nRows with F1_D = 0 or NaN: {len(null_D)}")
    print(f"Rows with F1_E = 0 or NaN: {len(null_E)}")


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 58, Finished, Available, Finished, False)

WARNING — these 5-class rows look degenerate (Macro-F1 < 0.25):
      paper             name    label  window  accuracy  macro_f1 weighted_f1  auc   auc_ovr f1_0 f1_1      f1_A      f1_B      f1_C      f1_D      f1_E runtime_sec family n_selected_feats
Alamri-2019     RandomForest y_5class       1  0.670451  0.247572    0.585175 None  0.600034 None None       0.0       0.0       0.0  0.787309  0.450549         6.7   None             None
Alamri-2019 GradientBoosting y_5class       1  0.670551   0.24947    0.585521 None   0.59922 None None   0.00885       0.0       0.0  0.787383  0.451117       23.86   None             None
Alamri-2019          XGBoost y_5class       1  0.670551  0.249495    0.585549 None  0.600871 None None  0.008811       0.0       0.0  0.787383  0.451282        9.71   None             None
Basnet-2022               NB y_5class       1  0.135843  0.147644    0.100193 None  0.650348 None None  0.021448  0.341324  0.084653  0.029282  0.261514        6.83     ML         

In [7]:
# ============================================================
# 4) LEADERBOARDS
# ============================================================
print(">>> BINARY LABEL — top 20 by Macro-F1")
b = (unified[unified["label"] == "y_binary"]
        .sort_values("macro_f1", ascending=False).head(20))
print(b.round(4).to_string(index=False))

print("\n>>> 5-CLASS LABEL — top 20 by Macro-F1")
m = (unified[unified["label"] == "y_5class"]
        .sort_values("macro_f1", ascending=False).head(20))
print(m.round(4).to_string(index=False))


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 60, Finished, Available, Finished, False)

>>> BINARY LABEL — top 20 by Macro-F1
       paper         name    label  window  accuracy  macro_f1 weighted_f1       auc auc_ovr      f1_0      f1_1 f1_A f1_B f1_C f1_D f1_E runtime_sec family n_selected_feats
Aouarib-2025       GA-XGB y_binary       4   0.93417  0.924656    0.933719  0.978541    None  0.951429  0.897884 None None None None None      729.03   None             26.0
Aouarib-2025       GA-XGB y_binary       2  0.931282  0.921165    0.930725  0.977381    None  0.949406  0.892924 None None None None None      788.43   None             17.0
Aouarib-2025     PlainXGB y_binary       4  0.930585  0.920188    0.929939  0.976219    None  0.948994  0.891382 None None None None None       10.41   None             None
Aouarib-2025     PlainXGB y_binary       2  0.926402  0.915337    0.925698  0.974896    None  0.945944  0.884729 None None None None None       15.81   None             None
   Dass-2021 RandomForest y_binary       4  0.926302  0.915102    0.925541  0.972849    None

In [8]:
# ============================================================
# 5) PER-WINDOW LEADERBOARD
# ============================================================
for W in [1, 2, 4]:
    print(f"\n========== W = {W} ==========")
    for label in ["y_binary", "y_5class"]:
        sub = (unified[(unified["window"] == W) & (unified["label"] == label)]
               .sort_values("macro_f1", ascending=False).head(8))
        if len(sub):
            cols = ["paper","name","accuracy","macro_f1","weighted_f1"]
            for c in ["f1_D","f1_E","f1_1","f1_0","runtime_sec"]:
                if c in sub.columns: cols.append(c)
            print(f"-- {label} (top 8) --")
            print(sub[cols].round(4).to_string(index=False))


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 62, Finished, Available, Finished, False)


========== W = 1 ==========
-- y_binary (top 8) --
       paper         name  accuracy  macro_f1 weighted_f1 f1_D f1_E      f1_1      f1_0 runtime_sec
Aouarib-2025       GA-XGB  0.924709  0.913339    0.923965 None None  0.881949  0.944729      892.75
Aouarib-2025     PlainXGB  0.923016  0.911368    0.922244 None None  0.879238  0.943498       26.84
   Dass-2021 RandomForest  0.921721  0.909567     0.92079 None None  0.876415   0.94272       34.64
 Basnet-2022           DT   0.90011  0.883878    0.898575 None None  0.840464  0.927292        9.72
 Basnet-2022          kNN  0.891545  0.874161    0.889994 None None  0.827389  0.920932      137.21
Xing&Du-2019      DeepMLP  0.891345  0.870605    0.888142 None None  0.818801   0.92241       86.87
 Basnet-2022          DNN  0.890051  0.870244    0.887405 None None  0.819549   0.92094        28.1
 Basnet-2022        CNN1D  0.883378  0.865123     0.88192 None None  0.815503  0.914743       40.56
-- y_5class (top 8) --
       paper         name

In [9]:
# ============================================================
# 6) HEAD-TO-HEAD vs EarlyRisk-MOOC (thesis Bảng 4)
# ============================================================
THESIS = pd.DataFrame([
    {"paper": "EarlyRisk-MOOC", "name": "KNN+BiLSTM+cb_focal_ord",
     "label": "y_5class", "window": 1,
     "accuracy": 0.8963, "macro_f1": 0.8460,
     "f1_D": 0.9280, "f1_E": 0.8415, "auc_ovr": 0.9874, "runtime_sec": 40.7},
    {"paper": "EarlyRisk-MOOC", "name": "KNN+BiLSTM+cb_focal",
     "label": "y_5class", "window": 2,
     "accuracy": 0.9686, "macro_f1": 0.9499,
     "f1_D": 0.9772, "f1_E": 0.9588, "auc_ovr": 0.9974, "runtime_sec": 48.9},
    {"paper": "EarlyRisk-MOOC", "name": "KNN+BiLSTM+cb_focal",
     "label": "y_5class", "window": 4,
     "accuracy": 0.9785, "macro_f1": 0.9556,
     "f1_D": 0.9847, "f1_E": 0.9813, "auc_ovr": 0.9979, "runtime_sec": 56.2},
])

best_by_W = (unified[unified["label"] == "y_5class"]
             .sort_values(["window","macro_f1"], ascending=[True, False])
             .groupby("window").head(1)
             .assign(source="baseline (this work, V4)"))

thesis_view = THESIS.copy().assign(source="thesis EarlyRisk-MOOC")

cols = ["source","paper","name","window","accuracy","macro_f1","f1_D","f1_E","runtime_sec"]
cols_b = [c for c in cols if c in best_by_W.columns]
cols_t = [c for c in cols if c in thesis_view.columns]
cmp = pd.concat([best_by_W[cols_b], thesis_view[cols_t]], ignore_index=True)
cmp = cmp.sort_values(["window","macro_f1"], ascending=[True, False])
print(">>> Best baseline at each W (5-class) vs EarlyRisk-MOOC")
print(cmp.round(4).to_string(index=False))


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 63, Finished, Available, Finished, False)

>>> Best baseline at each W (5-class) vs EarlyRisk-MOOC
                  source          paper                    name  window  accuracy  macro_f1      f1_D      f1_E runtime_sec
baseline (this work, V4)   Aouarib-2025                PlainXGB       1  0.918235  0.862846  0.947682  0.911344       22.68
   thesis EarlyRisk-MOOC EarlyRisk-MOOC KNN+BiLSTM+cb_focal_ord       1    0.8963     0.846     0.928    0.8415        40.7
   thesis EarlyRisk-MOOC EarlyRisk-MOOC     KNN+BiLSTM+cb_focal       2    0.9686    0.9499    0.9772    0.9588        48.9
baseline (this work, V4)   Aouarib-2025                  GA-XGB       2   0.92431  0.863767  0.953588  0.929199     1169.23
   thesis EarlyRisk-MOOC EarlyRisk-MOOC     KNN+BiLSTM+cb_focal       4    0.9785    0.9556    0.9847    0.9813        56.2
baseline (this work, V4)   Aouarib-2025                  GA-XGB       4  0.931083  0.865809  0.960634  0.944383      1570.1


In [10]:
# ============================================================
# 7) EXPORT
# ============================================================
unified.to_csv("/lakehouse/default/Files/result/all_baselines_combined.csv", index=False)
print("Saved combined results -> /lakehouse/default/Files/result/all_baselines_combined.csv")

summary = (unified
           .groupby(["paper","label","window"])
           .agg(best_macro_f1=("macro_f1","max"),
                best_acc=("accuracy","max"),
                fastest_rt=("runtime_sec","min"))
           .round(4).reset_index())
print("\n=== Per-paper × label × window summary ===")
print(summary.to_string(index=False))
summary.to_csv("/lakehouse/default/Files/result/all_baselines_summary.csv", index=False)
print("\nSaved summary -> /lakehouse/default/Files/result/all_baselines_summary.csv")


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 64, Finished, Available, Finished, False)

Saved combined results -> /lakehouse/default/Files/result/all_baselines_combined.csv

=== Per-paper × label × window summary ===
         paper    label  window best_macro_f1  best_acc fastest_rt
   Alamri-2019 y_5class       1      0.256933   0.67085        6.7
   Alamri-2019 y_5class       2      0.255635  0.669156       6.73
   Alamri-2019 y_5class       4      0.261169  0.667961       8.54
   Alamri-2019 y_binary       1      0.401401  0.669057       6.94
   Alamri-2019 y_binary       2      0.400859  0.669057       5.63
   Alamri-2019 y_binary       4      0.401148  0.669057       5.47
  Aouarib-2025 y_5class       1      0.862846  0.922916      22.68
  Aouarib-2025 y_5class       2      0.863767   0.92431      16.95
  Aouarib-2025 y_5class       4      0.865809  0.931083      21.27
  Aouarib-2025 y_binary       1      0.913339  0.924709      26.84
  Aouarib-2025 y_binary       2      0.921165  0.931282      15.81
  Aouarib-2025 y_binary       4      0.924656   0.93417      10.41
